# Stage 4 — Intervention analysis: where does ablating the latent direction matter?

Loads `results/intervention_results/` (now produced at multiple layer depths via the layer sweep), joins to baseline `results/logit_scores/`, and computes:

1. **Sanity-check audit** — per (model, attribute, method, layer) verify probe nullification + LM perplexity
2. **Headline delta table** — bias_baseline − bias_intervened per cell, with `layer_idx` / `depth_frac`
3. **Bias-delta vs depth curves** — the key new figure: where does the intervention actually reduce bias?
4. **Probe locus vs intervention locus dissociation** — the methodological headline claim
5. **Mechanism specificity** — ablating gender should drop gender-CrowS pairs more than race-color
6. **Variant × intervention regression** — does alignment depend on the latent direction more than base?

In [ ]:
import os
if 'COLAB_RELEASE_TAG' in os.environ:
    from google.colab import drive; drive.mount('/content/drive')
    %cd /content/llm-bias-eval
    if not os.path.lexists('results'):
        os.symlink('/content/drive/MyDrive/llm-bias-eval/results', 'results')

In [ ]:
from pathlib import Path
import pandas as pd

from biaseval.analysis.aggregate_results import (
    aggregate_logit_results, aggregate_probe_results, aggregate_intervention_results,
)

RESULTS = Path('results')
TABLES = Path('results/tables'); TABLES.mkdir(parents=True, exist_ok=True)

logit_df = aggregate_logit_results(RESULTS)
probe_df = aggregate_probe_results(RESULTS)
intv_df  = aggregate_intervention_results(RESULTS)
print(f'logit={len(logit_df)}  probe={len(probe_df)}  intervention={len(intv_df)}')
if not intv_df.empty:
    print(f"intervention layers per model: {intv_df.groupby('model_id')['layer_idx'].nunique().describe()}")

## 1. Sanity-check audit

Filter out any (model, attribute, method) cells where the intervention failed its sanity checks (probe didn't get nullified, or perplexity blew up). These shouldn't be interpreted as bias-mitigation evidence.

In [ ]:
if not intv_df.empty:
    audit = (intv_df[['model_id', 'attribute', 'method', 'layer_idx', 'depth_frac',
                       'probe_acc_post', 'probe_acc_passed',
                       'perplexity_ratio', 'perplexity_passed']]
             .drop_duplicates())
    failed = audit[~(audit['probe_acc_passed'].fillna(False) & audit['perplexity_passed'].fillna(False))]
    print(f'{len(failed)} of {len(audit)} (model, attr, method, layer) cells failed sanity:')
    print(failed.head(30))
    audit.to_csv(TABLES / 'intervention_sanity.csv', index=False)
    # Quick: which depths fail most? (Expect early layers — perplexity blow-up.)
    print('\nFail rate by depth (perplexity check):')
    print(audit.groupby('depth_frac')['perplexity_passed'].apply(
        lambda s: (~s.fillna(False)).mean()).round(3))

## 2. Headline before/after table

For each (model, benchmark, prompt_mode, attribute, method): bias_baseline (no intervention) vs bias_intervened, with delta. Headline metric per benchmark: CrowS overall stereo-win-rate, StereoSet SS.

In [ ]:
HEADLINE = {'crows_pairs': 'overall', 'stereoset': 'overall_SS'}

if intv_df.empty:
    delta_df = pd.DataFrame()
else:
    intv_keep = intv_df[intv_df.apply(lambda r: r['metric'] == HEADLINE.get(r['benchmark']), axis=1)].copy()
    intv_keep = intv_keep.rename(columns={'value': 'bias_intervened'})

    base = logit_df[logit_df.apply(lambda r: r['metric'] == HEADLINE.get(r['benchmark']), axis=1)].copy()
    base = base.rename(columns={'value': 'bias_baseline'})

    delta_df = intv_keep.merge(
        base[['model_id', 'benchmark', 'prompt_mode', 'bias_baseline']],
        on=['model_id', 'benchmark', 'prompt_mode'], how='left',
    )
    delta_df['bias_delta'] = delta_df['bias_baseline'] - delta_df['bias_intervened']
    cols = ['family', 'model_id', 'variant', 'benchmark', 'prompt_mode',
            'attribute', 'method', 'layer_idx', 'depth_frac',
            'bias_baseline', 'bias_intervened', 'bias_delta']
    delta_df = delta_df[cols].sort_values(cols[:7])
    delta_df.to_csv(TABLES / 'intervention_delta.csv', index=False)
delta_df.head(20)

## 3. Bias-delta vs intervention depth

Per (attribute × benchmark) facet, plot mean Δ bias as a function of intervention depth, separately for base vs instruct. The depth where the curve dips most is the **functional locus** of demographic representation — distinct from the layer where the probe peaks.

In [ ]:
from biaseval.analysis.plotting import (
    fig_intervention_by_layer, fig_probe_vs_intervention_loci,
)
FIGURES = Path('figures'); FIGURES.mkdir(exist_ok=True)

if not intv_df.empty and not delta_df.empty:
    # 9 — bias delta by depth, faceted by attribute × benchmark
    p = fig_intervention_by_layer(logit_df, intv_df, FIGURES)
    print('wrote', p)

# 10 — probe locus vs intervention locus (the methodological headline)
if not probe_df.empty and not intv_df.empty:
    p = fig_probe_vs_intervention_loci(probe_df, intv_df, logit_df, FIGURES)
    print('wrote', p)

## 4. Mechanism check — does ablating gender selectively reduce gender-CrowS pairs?

Per-bias-type breakdown of intervention effect. If the ablation is causally specific, ablating *gender* should drop the **gender** category of CrowS-Pairs more than e.g. **race-color** or **religion**.

In [ ]:
if not intv_df.empty:
    # CrowS-Pairs has per-bias-type metrics in its summary (e.g. 'gender', 'race-color', ...)
    bias_types = ['gender', 'race-color', 'religion', 'age', 'disability',
                  'physical-appearance', 'sexual-orientation', 'nationality', 'socioeconomic']
    crows_intv = intv_df[(intv_df['benchmark'] == 'crows_pairs') & intv_df['metric'].isin(bias_types)]
    crows_base = logit_df[(logit_df['benchmark'] == 'crows_pairs') & logit_df['metric'].isin(bias_types)]
    crows_intv = crows_intv.rename(columns={'value': 'intervened', 'metric': 'bias_type'})
    crows_base = crows_base.rename(columns={'value': 'baseline', 'metric': 'bias_type'})
    mech = crows_intv.merge(
        crows_base[['model_id', 'prompt_mode', 'bias_type', 'baseline']],
        on=['model_id', 'prompt_mode', 'bias_type'], how='left',
    )
    mech['delta'] = mech['baseline'] - mech['intervened']
    mech_summary = (mech.groupby(['attribute', 'method', 'bias_type'])['delta']
                    .mean().unstack('bias_type'))
    mech_summary.to_csv(TABLES / 'intervention_mechanism_check.csv')
    print(mech_summary.round(2))

## 5. Variant × intervention interaction (regression)

Does the intervention reduce bias *more* for instruct models than base? If yes, evidence that alignment was relying on the latent direction.

In [ ]:
if not delta_df.empty and len(delta_df) > 10:
    import statsmodels.formula.api as smf
    m = smf.ols(
        "bias_delta ~ C(variant) + C(method) + C(attribute) + C(benchmark) "
        "+ C(prompt_mode) + depth_frac + C(variant):depth_frac",
        data=delta_df,
    ).fit(cov_type='cluster', cov_kwds={'groups': delta_df['model_id'].values})
    print(m.summary())